In [1]:
!pip install wandb -qU
import wandb

In [2]:
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: karanshelar8775 (karanshelar8775-student-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
import wandb

# Initialize the tracking project
wandb.init(
    project="llama3-lora-finetuning",
    name="alpaca-dolly-oasst-run-1"
)

In [4]:
!pip install --quiet --upgrade "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --quiet --no-deps "xformers<0.0.35" "trl" peft accelerate bitsandbytes

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 136.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 130.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 127.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.8/110.8 MB 8.5 MB/

In [5]:
from datasets import load_dataset
import pandas as pd

In [7]:
alpaca = load_dataset("json", data_files="/content/alpaca_dataset.json", split="train")

dolly = load_dataset("json", data_files="/content/dolly_dataset.jsonl", split="train")

oasst = load_dataset("parquet", data_files="/content/openassistant_dataset.parquet", split="train")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [8]:
print(alpaca.column_names)
print(dolly.column_names)
print(oasst.column_names)

['instruction', 'input', 'output']
['instruction', 'context', 'response', 'category']
['instruction', 'input', 'output', 'text']


In [9]:
def convert_dolly(example):
  return{
      "instruction":example["instruction"],
      "input":example["context"] if example["context"] else "",
      "output":example["response"]
  }
dolly = dolly.map(convert_dolly)

Map:   0%|          | 0/15011 [00:00<?, ? examples/s]

In [10]:
oasst = oasst.remove_columns(["text"])

In [11]:
alpaca = alpaca.select(range(5000))
dolly = dolly.select(range(5000))
oasst = oasst.select(range(5000))

In [12]:
from datasets import concatenate_datasets

merged_dataset = concatenate_datasets([alpaca, dolly, oasst])

In [13]:
print(len(merged_dataset))

15000


In [14]:
def format_prompt(example):
    return {
        "text": f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}"""
    }

merged_dataset = merged_dataset.map(format_prompt)

Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

In [15]:
print(merged_dataset[0]["text"])

### Instruction:
Give three tips for staying healthy.

### Input:


### Response:
1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.

2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.

3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.


In [16]:
import torch
import gc
from unsloth import FastLanguageModel

# Clear memory fragments
gc.collect()
torch.cuda.empty_cache()

max_seq_length = 2048
dtype = None          # Auto-selects Float16 for Tesla T4
load_in_4bit = True   # Mandatory for free-tier GPUs

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


In [18]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Saves massive amounts of VRAM
    random_state = 3407,
)

Unsloth 2026.3.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [19]:
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

# 1. Prepare model for training
FastLanguageModel.for_training(model)

# 2. Define Professional Training Arguments
args = TrainingArguments(
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 8,
    warmup_steps = 10,
    max_steps = 300,             # Keep at 300 for a full run, or 60 for a quick test
    learning_rate = 1e-4,
    fp16 = True,
    bf16 = False,
    logging_steps = 1,           # <--- CHANGED TO 1: See live updates on WandB every step!
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "outputs",

    # === WANDB SETTINGS ===
    report_to = "wandb",         # Enforces tracking to WandB
    run_name = "llama3-lora-final-run"
)

# 3. Initialize Trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = merged_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = args,
)

# 4. Start Training & Watch the magic!
trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/15000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 15,000 | Num Epochs = 1 | Total steps = 300
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
1,1.838078
2,1.680609
3,1.973745
4,2.195661
5,2.198287
6,1.987798
7,2.300113
8,1.586008
9,1.483245
10,1.890962


train/epoch,▁▁▁▁▂▂▂▂▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▅▆▆▇▇▇▇▇▇▇▇▇█████
train/global_step,▁▁▁▁▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▆▆▆▆▇▇▇▇▇▇████
train/grad_norm,▂█▂▂▃▂▂▂▂▁▁▂▁▁▁▁▁▂▁▁▁▂▂▂▁▁▂▁▂▂▁▁▁▂▁▂▁▁▂▁
train/learning_rate,▁▅█▇▇▇▇▇▇▇▇▆▆▆▆▆▆▆▆▅▅▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▁
train/loss,▄▂▅▅▅▄▃▃▃▂▅▄▁▃▄▂▄█▄▂▃▃▃▄▄▁▃▃▁▄▂▂▂▃▁▃█▁▃▅
total_flos,1.484068894961664e+16
train/epoch,0.16
train/global_step,300
train/grad_norm,0.61807
train/learning_rate,0.0
train/loss,1.80642


In [20]:
# 1. Save results locally in Colab
model.save_pretrained("lora_model_final")
tokenizer.save_pretrained("lora_model_final")

# 2. Finish the WandB run
import wandb
wandb.finish()

# 3. Zip and Download to your PC
import os
from google.colab import files
os.system("zip -r lora_model_final.zip lora_model_final")
files.download("lora_model_final.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>